# Lab of Week 4: Probabalistic Machine Learning

Today we will look at some foundations of probabilistic machine learning. Namely, these topics:

* Kernel Smoothing and Nadaraya-Watson Regression
* Gaussian Processes (for Regression)
* Uncertainty and Calibration
* Approximate Bayesian Inference

In this lab, we will build intuition for how probabilistic models capture uncertainty in data and predictions. Starting from kernel smoothing and the Nadaraya–Watson estimator, we will see how simple local averaging connects to more powerful Bayesian approaches such as Gaussian Processes, which generalize these ideas. We will then explore the role of uncertainty estimates in practice, looking at when models can be overconfident and how calibration methods help correct this. Finally, we will touch on approximate Bayesian inference techniques, which make it possible to apply these ideas at scale when exact inference is intractable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

## Kernel Smoothing and Nadaraya-Watson Regression

Kernel smoothing is a nonparametric regression method: instead of assuming a fixed functional form for $f(x)$, it estimates the value at a point by averaging nearby observations, with closer points weighted more heavily. The Nadaraya–Watson estimator formalizes this idea.

First we'll generate some data: $x \sim p(x)$ and $y \sim p(y|x)$. 

In [ ]:
np.random.seed(0)
n = 30
x = np.sort(np.random.uniform(0, 10, size=n))
y = np.sin(x) + 0.3 * np.random.randn(n)

x_plot = np.linspace(0, 10, 500)
plt.figure()
plt.scatter(x,y)
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.tight_layout()

## Have a look at some kernels

The kernel function is at the heart of kernel smoothing, since it determines how distances are translated into weights. A Gaussian kernel, for example, assigns smoothly decaying weights as we move further from the evaluation point, while compact kernels like the top-hat or Epanechnikov assign nonzero weight only within a fixed window. Each choice embodies a different trade-off between smoothness and locality, and experimenting with them helps build intuition for how "local averaging" can be tuned to fit data patterns.

In [ ]:
def gaussian_kernel(u):
    return (1/np.sqrt(2*np.pi)) * np.exp(-0.5 * u**2)

def top_hat_kernel(u):
    return 0.5 * ((u >= -1) & (u <= 1)).astype(float)

def epanechnikov_kernel(u):
    return 0.75 * (1 - u**2) * ((u >= -1) & (u <= 1)).astype(float)

## Implementation

In practice, the Nadaraya–Watson estimator is implemented by computing a weighted average. This requires normalizing the kernel weights so they sum to one, ensuring the result is an average rather than a raw sum. 

### Task

Implement Nadaraya-Watson regression from scratch in the function `nwr` below, where `kernel_fn` refers to one of the kernel functions defined above. See the cell after to get an intuition as to what is expected as a return type. Hint: do not modify `x`, `y`. When you've finished, try for different `kernel_fn`s and different `h` and see what happens.


In [ ]:
def nwr(x, y, x_eval, h, kernel_fn=gaussian_kernel):
    """
    Compute the Nadaraya–Watson regression estimate at given evaluation points.

    Parameters
    ----------
    x : array-like, shape (n,)
        Training points
    y : array-like, shape (n,)
        Corresponding target values.
    x_eval : array-like, shape (n_eval,)
        Points at which to evaluate the regression estimate.
    h : float
        Bandwidth parameter for the kernel.
    kernel_fn : callable, default=gaussian_kernel
        Kernel function applied to scaled distances (must accept array input).

    Returns
    -------
    f_hat : ndarray, shape (n_eval,)
        Estimated regression values at each point in x_eval.
    """
    ### BEGIN SOLUTION
    # ensure proper shapes
    x = x[:, None]          # shape (n, 1)
    y = y[:, None]          # shape (n, 1)
    x_eval = x_eval[None, :]  # shape (1, n_eval)

    # compute scaled distances (n.b. bandwith parameter h)
    u = (x_eval - x) / h    

    # apply kernel -> weights
    K = kernel_fn(u)        

    # the famous weighted average
    f_hat = (K * y).sum(axis=0) / K.sum(axis=0)  # shape (n_eval,)
    return f_hat
    ### END SOLUTION

In [ ]:
# Test your implementation
f_hat = nwr(x, y, x_plot, h=0.5, kernel_fn=gaussian_kernel)

plt.figure()
plt.scatter(x, y, c='k', label='$(x_i,y_i)$')
plt.plot(x_plot, f_hat, 'g', label=r"$\hat f$")
plt.title("Nadaraya-Watson Regression")
plt.legend()
plt.tight_layout()

## Gaussian Processes 

A Gaussian distribution describes a random variable. 
$$
    N(\mathbf{0},\mathbf{\Sigma})
$$

A Gaussian Process generalizes this idea to random functions. Instead of placing a distribution over vectors, we place a distribution over function values evaluated at any finite set of inputs. In practice, this means that for inputs $X = \{x_1,\ldots,x_n\}$, the corresponding function values follow a joint Gaussian distribution: 
$$
    N(\mathbf{0},\mathbf{K}_X)
$$
where $\mathbf{K}_X$ is the covariance matrix defined by a kernel function. 
Complete the following code, using the given kernel function. Hint: if `X` if of length $n$, then `K` is of dimension $n \times n$; 
`size` is simply the number of random functions we want to sample.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

### Task 

Complete the code below, to provide `K`, using the specified `kernel`. 

In [ ]:
# GP prior
X = np.linspace(-5, 5, 100).reshape(-1, 1)
kernel = 1.0 * RBF(length_scale=1.0)
### BEGIN SOLUTION
K = kernel(X, X)
### END SOLUTION
samples = np.random.multivariate_normal(mean=np.zeros(len(X)), cov=K, size=5)

plt.figure()
for i in range(5):
    plt.plot(X, samples[i])
plt.title("Samples from GP Prior")
plt.xlabel('$x$')
plt.ylabel('$f(x)$')
plt.show()

### Task 

Have a look at the functionality of `GaussianProcessRegressor` in Scikit-Learn, and complete the missing line in the following
(the model `gpr` is already fit for you; check the documentation and understand how to extract the mean and standard deviation of the predictive distribution)

In [ ]:
kernel = RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1)
gpr = GaussianProcessRegressor(kernel=kernel, alpha=0.0)
gpr.fit(x.reshape(-1,1), y)

# mu, std = ...
### BEGIN SOLUTION
mu, std = gpr.predict(x_plot.reshape(-1,1), return_std=True)
### END SOLUTION

plt.figure()
plt.scatter(x, y, c='k', label='Data')
plt.plot(x_plot, mu, 'b', label='GPR mean')
plt.fill_between(x_plot, mu - 2*std, mu + 2*std, alpha=0.2, label='95% CI')
plt.plot(x_plot, f_hat, 'g', label='NWR')
plt.plot(x_plot, np.sin(x_plot), 'k--', label='Truth')
plt.title("GPR vs NWR (1D)")
plt.legend()
plt.tight_layout()

## Uncertainty Analysis

### Task: Reliability Diagram

Provide a reliability diagram for GPR, by completing the code below (fill `empirical_coverages`). When you've fininshed, think abouth the limitations of such a comparison. 

In [ ]:
# Recompute predictive mean and std at the training points
mu_train, std_train = gpr.predict(x[:, None], return_std=True)

# Nominal coverage levels
coverages = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
empirical_coverages = []

### BEGIN SOLUTION
from scipy.stats import norm
for c in coverages:
    # Convert nominal coverage to z-score, e.g. 95% -> 1.96
    z = norm.ppf(0.5 + c/2)
    lower = mu_train - z * std_train
    upper = mu_train + z * std_train
    # Proportion of true y inside interval
    inside = ((y >= lower) & (y <= upper)).mean()
    empirical_coverages.append(inside)
### END SOLUTION

In [ ]:
# Plot reliability diagram for regression
plt.figure()
plt.plot(coverages, empirical_coverages, 'o-', label="GPR")
plt.plot([0, 1], [0, 1], 'k--', label="Ideal")
plt.xlabel("Nominal coverage")
plt.ylabel("Empirical coverage")
plt.title("Calibration curve (regression)")
plt.legend()
plt.show()

In [ ]:
### BEGIN SOLUTION
# Limitations: 
# * We didn't use internal validation
# * Perhaps too few points (especialyl, per bin) 
### END SOLUTION

## GPR with multi-dimensional data.

i.e., when $\textbf{x} \in \mathbb{R}^d$. 

First we will generate some data

In [ ]:
from sklearn.gaussian_process.kernels import RBF

def f_2d(X):
    return np.sin(X[:, 0]) + np.cos(3 * X[:, 1]) + 0.3 * X[:, 0] * X[:, 1]

n2 = 100
X2 = np.random.uniform(0, 5, size=(n2, 2))
y2 = f_2d(X2) + 0.2 * np.random.randn(n2)

### Task: Gaussian Process for dimension d=2

Complete the following code where indicated (get the elements of the predictive distribution from the `gpr2d` model, which has already been fit for you). 

In [ ]:
# Fit GPR
gpr2d = GaussianProcessRegressor(kernel=RBF(length_scale=1.0) + WhiteKernel(0.1))
gpr2d.fit(X2, y2)

# Grid for prediction
xx, yy = np.meshgrid(np.linspace(0, 5, 50), np.linspace(0, 5, 50))
Xtest = np.column_stack([xx.ravel(), yy.ravel()])

### BEGIN SOLUTION
mu2d, std2d = gpr2d.predict(Xtest, return_std=True)
### END SOLUTION
# mu2d = ..
# std2d = ..

In [ ]:
plt.figure()
plt.contourf(xx, yy, mu2d.reshape(xx.shape), cmap=cm.viridis)
plt.colorbar()
plt.scatter(X2[:, 0], X2[:, 1], c='r', s=10, label='Train data')
plt.title("GPR mean prediction (2D)")
plt.tight_layout()
plt.legend()

plt.show()